In [46]:
using LowLevelFEM, LinearAlgebra

In [47]:
openGeometry("boxes.geo")

In [48]:
#openPreProcessor()

In [49]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [50]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 4250)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

u = solveField(K, f, support=[bc_bottom, bc_top])

#showDoFResults(u, name="u", factor=1, visible=true)

nodal VectorField
[0.0; 0.0; … ; -0.02164690521873969; -0.006711214389714968;;]


## Lagrange multiplier contact with mixed surface interpolation

The displacement/contact-gap field and the Lagrange multiplier do not need to
use the same interpolation order. Here

$$
u_h,\; g_h \in P_2,
\qquad
\lambda_h \in P_1
$$

because the multiplier field is created with `reducedOrder=true`.

The contact geometry remains described by

$$
G:V_u\rightarrow V_c,
\qquad
d=G(r+u).
$$

For frictionless contact only the active normal component is used. Let

$$
P_n:V_c\rightarrow V_{n,a}
$$

select the active normal contact components.

The surface coupling is assembled directly as a mixed bilinear form,

$$
M_{\lambda u}
=
\int_{\Gamma_s}
N_\lambda^T\,D_n\,N_u\,d\Gamma,
$$

where the test interpolation is provided by the reduced-order multiplier field
and the trial interpolation by the displacement field. This is the important
difference from extracting a $P_2/P_2$ surface matrix after assembly.

After restriction to the slave surface and active normal contact space,

$$
M_a=P_nM_{\lambda u}P_n^T,
\qquad
G_n=P_nG,
\qquad
E_n=EP_n^T.
$$

The global contact constraint operator is

$$
B=E_nM_aG_n.
$$

The KKT residuals are

$$
r_u=Ku-f+B^T\lambda,
\qquad
r_\lambda=E_nM_a d_{n,a}.
$$


In [51]:

using SparseArrays

r = nodePositionVector(U)

# The multiplier field has the same number of components as U.
# On the slave contact surface these components are interpreted in the
# local contact basis (n,t). For frictionless contact only the first
# component is used.
Λ = Field([mat], type=:VectorField, dim=3, fieldName=:λ, reducedOrder=true)

println("contact")
@time L = contact(
    u,
    master="master",
    slave="slave",
    LagrangeMultiplierField=Λ,
    topology_tol=0.01
)

support = [bc_bottom, bc_top]
free_u = freeDoFs(U, support)

u_it = copy(u)
λ_it = vectorField(Λ, "body", [0, 0, 0])

# Zero multiplier block of the KKT system.
Zλ = SystemMatrix(spzeros(ndofs(Λ), ndofs(Λ)), Λ)

# PDAS scaling parameter; this is not a penalty stiffness.
κ = 1e7;


contact
  0.380645 seconds (176.90 k allocations: 18.985 MiB)



### Active normal selection

`L.Pa` selects complete local contact blocks $(n,t_1,t_2)$. For frictionless
Lagrange contact only the normal component is constrained, so the helper below
constructs

$$
P_n:V_c\rightarrow V_{n,a}.
$$

The same selector is used on both sides of the mixed surface matrix because the
multiplier and displacement fields are stored on the same mesh-node ordering.
Their interpolation orders, however, are different and are handled by the
mixed `∫(Λ ⋅ Dn ⋅ U, ...)` assembly itself.


In [52]:

function activeNormalSelection(
    C::Contact,
    active::AbstractVector{Bool}
)
    length(active) == length(C.slave_nodes) ||
        error("activeNormalSelection: incompatible active-set size.")

    pdim = C.U.pdim
    active_nodes = findall(active)
    na = length(active_nodes)

    rows = collect(1:na)
    cols = (active_nodes .- 1) .* pdim .+ 1
    vals = ones(Float64, na)

    P = sparse(rows, cols, vals, na, length(C.d))

    return SystemMatrix(
        P,
        nothing,
        nothing,
        nothing,
        nothing
    )
end


activeNormalSelection (generic function with 1 method)


The sign convention used below is

$$
g_n\ge 0 \quad\text{open/admissible},
\qquad
\lambda_n\le 0 \quad\text{compression}.
$$

The primal-dual active-set rule is therefore

$$
\lambda_n+\kappa g_n<0.
$$

Inactive normal multipliers are set to zero. Tangential multipliers are never
included among the free multiplier DoFs, so they remain zero automatically.


In [53]:
pdim = L.U.pdim
nu = ndofs(U)

# ----------------------------------------------------------
# Full normal contact-space selector
#
# Pn0 keeps the normal component of every candidate
# contact node. The active set will NOT be applied here:
# it will live in the reduced P1 multiplier space.
# ----------------------------------------------------------
all_contact = trues(length(L.slave_nodes))

Pn0 = activeNormalSelection(
    L,
    all_contact
)

# ----------------------------------------------------------
# Consistent scalar surface integration matrix
#
# We only use one component block of the vector surface
# matrix. This gives the scalar surface mass matrix
#
#     Mn_ij = ∫ Ni Nj dΓ
#
# in the contact-node ordering.
# ----------------------------------------------------------
Dn = zeros(Float64, pdim, pdim)
Dn[1, 1] = 1.0

M0 = ∫(
    U ⋅ Dn ⋅ U,
    Γ="slave"
)

Mc = subSystemMatrix(
    M0;
    onPhysicalGroup="slave"
)

Mn = Pn0 * Mc * Pn0'

# ----------------------------------------------------------
# Embedding of the complete normal contact space into the
# full multiplier field.
#
#     En : Vn -> Vλ(full)
# ----------------------------------------------------------
En = L.E * Pn0'

# ----------------------------------------------------------
# Reduced-order multiplier transformation
#
#     λfull = Tλ λr
#     λr    = Rλ λfull
# ----------------------------------------------------------
Tλ, Rλ = reductionMatrices(Λ)

nλr = size(Tλ, 2)

# ----------------------------------------------------------
# Map nodal normal contact quantities to the reduced P1
# multiplier coordinates.
#
# Qr is used only for the pointwise PDAS criterion.
# ----------------------------------------------------------
Qr = Rλ * En.A

Iλ, _, _ = findnz(Qr)

candidate_λr =
    sort!(
        unique!(Iλ)
    )

println(
    "reduced multiplier dofs = ", nλr,
    ", contact multiplier dofs = ", length(candidate_λr)
)

# Reduced multiplier is the actual unknown.
λr_it =
    Rλ * DoFs(λ_it)[:, 1]

# Always keep the stored full field exactly inside the
# reduced-order space.
DoFs(λ_it)[:, 1] .= Tλ * λr_it

active_old =
    falses(length(candidate_λr))

active_old2 =
    falses(length(candidate_λr))


for iter in 1:40

    # ==========================================================
    # Current contact geometry
    # ==========================================================
    println("updateContact")
    @time updateContact!(L, u_it)

    # ----------------------------------------------------------
    # Normal contact kinematics on ALL candidate slave nodes
    #
    # Gn : Vu -> Vn
    # dn : current normal gap vector
    # ----------------------------------------------------------
    Gn = Pn0 * L.G
    dn = Pn0 * L.d

    # ----------------------------------------------------------
    # Reduced P1 values used by PDAS
    #
    # gap_r contains the normal gap evaluated in the reduced
    # multiplier coordinates.
    # ----------------------------------------------------------
    gap_r = Qr * dn.a

    λc =
        λr_it[candidate_λr]

    gc =
        gap_r[candidate_λr]

    # ----------------------------------------------------------
    # Primal-dual active set in the REDUCED multiplier space
    #
    # g >= 0       open/admissible
    # λ <= 0       compression
    #
    # active <=> λ + κ g < 0
    # ----------------------------------------------------------
    active =
        λc .+ κ .* gc .< 0.0

    # ----------------------------------------------------------
    # Detect A -> B -> A cycling
    # ----------------------------------------------------------
    #=
    two_cycle =
        iter > 2 &&
        active == active_old2 &&
        active != active_old

    if two_cycle

        switching =
            active .!= active_old

        println(
            "Two-cycle detected: freezing ",
            count(switching),
            " switching reduced multiplier DOFs."
        )

        active[switching] .=
            active_old[switching]
    end
    =#
    
    active_rows =
        candidate_λr[active]

    inactive_rows =
        candidate_λr[.!active]

    # Inactive multipliers vanish IN THE REDUCED SPACE.
    λr_it[inactive_rows] .= 0.0

    # ==========================================================
    # Consistent LM weak form
    #
    # Full multiplier representation:
    #
    #     Bfull = En Mn Gn
    #
    #     gfull = En Mn dn
    #
    # Then Galerkin projection to the P1 multiplier space:
    #
    #     Br = Tλ' Bfull
    #     gr = Tλ' gfull
    # ==========================================================
    println("Bfull")
    @time Bfull =
        En * Mn * Gn

    println("gfull")
    @time gfull =
        En * (Mn * dn)

    println("reduction")
    @time Br =
        Tλ' * Bfull.A

    gr =
        Tλ' * gfull.a[:, 1]

    # ----------------------------------------------------------
    # Only ACTIVE reduced multiplier equations enter the KKT
    # system.
    # ----------------------------------------------------------
    Ba =
        Br[active_rows, :]

    ga =
        gr[active_rows]

    na =
        length(active_rows)

    # ==========================================================
    # Mechanical residual
    #
    # ru = K u - f + Br' λr
    #
    # λr contains zero values on inactive multiplier DOFs.
    # ==========================================================
    println("r_u")
    @time r_u =
        K * u_it - f

    r_u.a[:, 1] .+=
        Br' * λr_it

    # ==========================================================
    # Active KKT system
    #
    #       [ K   Ba' ] [Δu ]   [ru]
    #       [ Ba   0  ] [Δλa] =-[ga]
    #
    # Only active P1 multiplier DOFs appear in the system.
    # ==========================================================
    println("A")

    @time A =
        [
            K.A   Ba'
            Ba    spzeros(na, na)
        ]

    res =
        vcat(
            r_u.a[:, 1],
            ga
        )

    # Prescribed displacement DOFs receive zero Newton
    # increment because only free_u is included.
    free =
        vcat(
            free_u,
            nu .+ collect(1:na)
        )

    Δx =
        zeros(Float64, nu + na)

    println("solve")

    @time Δx[free] =
        -A[free, free] \ res[free]

    Δu =
        @view Δx[1:nu]

    Δλa =
        @view Δx[nu+1:end]

    # ==========================================================
    # Update
    # ==========================================================
    DoFs(u_it)[:, 1] .+=
        Δu

    λr_it[active_rows] .+=
        Δλa

    # CRITICAL:
    # never modify individual full-order multiplier nodes.
    #
    # The full field is always reconstructed from the P1
    # reduced coordinates.
    DoFs(λ_it)[:, 1] .=
        Tλ * λr_it

    # ----------------------------------------------------------
    # Post-update contact state
    # ----------------------------------------------------------
    updateContact!(L, u_it)

    Gn_check = Pn0 * L.G
    dn_check = Pn0 * L.d

    gap_r_new = Qr * dn_check.a
    
    λc_new = λr_it[candidate_λr]
    gc_new = gap_r_new[candidate_λr]
    
    active_new =
        λc_new .+ κ .* gc_new .< 0.0
    
    # ----------------------------------------------------------
    # Complementarity diagnostics
    #
    # g >= 0
    # λ <= 0
    # λ*g = 0
    # ----------------------------------------------------------
    p_new = -λc_new
    
    gap_violation =
        max(
            0.0,
            -minimum(gc_new)
        )
    
    pressure_violation =
        max(
            0.0,
            -minimum(p_new)
        )
    
    complementarity =
        maximum(
            abs.(p_new .* gc_new)
        )
    
    err_u =
        norm(Δu[free_u]) /
        max(
            norm(DoFs(u_it)[free_u]),
            eps()
        )
    
    Δactive =
        count(active_new .!= active)
    
    println(
        "iter = ", iter,
        ", active = ", count(active_new),
        ", Δactive = ", Δactive,
        ", min gap = ", minimum(gc_new),
        ", min pressure = ", minimum(p_new),
        ", complementarity = ", complementarity,
        ", error = ", err_u
    )
    
    converged =
        active_new == active &&
        gap_violation < 1e-8 &&
        pressure_violation < 1e-8 &&
        err_u < 1e-8
    
    active_old2 .= active_old
    active_old .= active_new
    
    converged && break
    
end


u_LM = u_it

# λ_it is already the P1 multiplier prolonged to the
# original full nodal representation.
λ_LM = λ_it

# Synchronize final geometry.
updateContact!(L, u_LM)

reduced multiplier dofs = 11961, contact multiplier dofs = 510
updateContact
  0.390107 seconds (178.58 k allocations: 19.105 MiB, 20.06% gc time)
Bfull
  0.001981 seconds (27 allocations: 6.075 MiB)
gfull
  0.000303 seconds (36 allocations: 1.821 MiB)
reduction
  0.002436 seconds (32 allocations: 5.686 MiB)
r_u
  0.010058 seconds (30 allocations: 1.806 MiB)
A
  0.329857 seconds (183 allocations: 425.601 MiB, 29.34% gc time)
solve
  4.715548 seconds (165 allocations: 1.159 GiB, 2.05% gc time)
iter = 1, active = 72, Δactive = 13, min gap = -0.002516021165306071, min pressure = -1877.00970315849, complementarity = 4.722596140631627, error = 0.048588113084997726
updateContact
  0.288033 seconds (178.47 k allocations: 20.728 MiB)
Bfull
  0.003604 seconds (27 allocations: 11.144 MiB)
gfull
  0.000212 seconds (18 allocations: 1.820 MiB)
reduction
  0.003546 seconds (26 allocations: 9.815 MiB)
r_u
  0.010822 seconds (18 allocations: 1.805 MiB)
A
  0.281116 seconds (121 allocations: 426.053 Mi

Contact("slave" -> "master", 1957 candidate nodes, 228 active, G=(5871, 78834), Pa=(684, 5871), Lagrange multiplier)

In [54]:

showDoFResults(u_LM, name="u LM", factor=1, visible=true)


0


## Contact gap and pressure

`L.d` is the reduced contact vector. Mapping it back to the displacement mesh
gives the ordinary LLFEM field representation, so the normal gap is simply the
first component.

With the sign convention used above the normal Lagrange multiplier is negative
in compression, therefore the positive contact pressure is

$$
p=-\lambda_n.
$$


In [55]:

DD = VectorField(L.d)
gap = DD[1]

pressure = -λ_LM[1]

showElementResults(nodesToElements(pressure, onPhysicalGroup="slave"), name="p")
showElementResults(nodesToElements(gap, onPhysicalGroup="slave"), name="gap")


2

In [56]:

# Optional postprocessing:
# showElementResults(
#     nodesToElements(pressure, onPhysicalGroup="slave"),
#     name="pressure",
#     visible=true
# )
#
# showElementResults(
#     nodesToElements(gap, onPhysicalGroup="slave"),
#     name="gap",
#     visible=true
# )

openPostProcessor()
